# LLM Evaluation — From Metrics to Custom Eval Pipelines

**Session 7 — Applied Natural Language Processing**

Good prompting is only half the picture. How do we know if our prompts actually work?

This notebook covers three parts:
- **Part 1 — Classical metrics**: BLEU, ROUGE, BERTScore, perplexity, and an end-to-end summarisation eval pipeline
- **Part 2 — Benchmarks & Leaderboards**: Guided reading — what to look for and what to question
- **Part 3 — Custom eval pipelines**: LLM-as-judge, DIY eval loop, Promptfoo, and the OpenAI Evals API

**Parts 1a–1c** require no API key. **Part 1d and Part 3** require a GitHub token (set `GITHUB_TOKEN` env var). **Part 3d** also requires an `OPENAI_API_KEY`.

In [1]:
# Part 1 dependencies — no API key needed
# Uncomment to install if not already in your environment:
# !pip install evaluate bert-score transformers torch

import warnings
warnings.filterwarnings("ignore")

import evaluate
from bert_score import score as bert_score_fn
import torch
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

from dotenv import load_dotenv
import os

load_dotenv()

# Load metrics once — reused throughout Part 1
bleu_metric = evaluate.load("bleu")
rouge_metric = evaluate.load("rouge")

print("Part 1 dependencies ready.")

Part 1 dependencies ready.


In [2]:
# GitHub Models client — required for Part 1d and Part 3
# Needs GITHUB_TOKEN set in your environment (free with a GitHub account)
from openai import OpenAI
import os

client = OpenAI(
    base_url="https://models.inference.ai.azure.com",
    api_key=os.environ.get("GITHUB_TOKEN", "set-your-github-token"),
)
MODEL = "gpt-4.1-nano"

def chat(prompt, system=None, temperature=0.3, max_tokens=300):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    response = client.chat.completions.create(
        model=MODEL, messages=messages,
        temperature=temperature, max_tokens=max_tokens,
    )
    return response.choices[0].message.content.strip()

print(chat("Reply with: ready"))
print("GitHub Models client ready.")

ready
GitHub Models client ready.


---

## Part 1 — Classical Metrics

Classical NLP metrics predate LLMs but remain widely used. They measure surface-level or semantic similarity between a model's output and a reference (gold-standard) text.

Sections 1a–1c use three hand-crafted summarisation examples:
- **Good summary**: captures key points, well-phrased
- **Bad summary**: misses key points, poor quality
- **Synonym summary**: correct meaning, different words — tests whether the metric penalises paraphrasing

Section 1d puts it all together: generate a real dataset, produce summaries, and score them with all three metrics.

### 1a — BLEU and ROUGE

**BLEU** (Bilingual Evaluation Understudy) measures n-gram precision — how many n-grams in the model output appear in the reference. Designed for machine translation.

**ROUGE** (Recall-Oriented Understudy for Gisting Evaluation) measures n-gram recall — how many reference n-grams appear in the model output. Designed for summarisation.

Both suffer from the same limitation: they measure surface overlap, not meaning. A correct paraphrase earns zero credit.

In [3]:
# Source article and reference summary
source = (
    "The city council voted unanimously to approve a new public transport initiative. "
    "The plan includes expanding the metro network by 12 new stations, introducing "
    "electric buses on all major routes, and building 50 new cycling lanes. "
    "The project is expected to reduce car usage by 30% over five years."
)
reference = (
    "The council approved a transport plan to expand metro, add electric buses, "
    "and build cycling lanes, aiming to cut car use by 30%."
)

# Three candidate summaries to compare
candidates = {
    "Good summary":    "The city council approved a transport initiative to expand the metro by 12 stations, introduce electric buses, and add cycling lanes, targeting a 30% reduction in car use.",
    "Bad summary":     "The council had a meeting and discussed various city matters including transport and other infrastructure topics.",
    "Synonym summary": "The municipal council unanimously endorsed a transit scheme to broaden the subway network, deploy electric coaches, and create bicycle paths, aiming to decrease vehicle traffic by 30%.",
}

print(f"{'Summary':<20} {'BLEU':>8} {'ROUGE-1':>10} {'ROUGE-L':>10}")
print("-" * 52)

for name, candidate in candidates.items():
    bleu = bleu_metric.compute(predictions=[candidate], references=[[reference]])
    rouge = rouge_metric.compute(predictions=[candidate], references=[reference])
    print(f"{name:<20} {bleu['bleu']:>8.3f} {rouge['rouge1']:>10.3f} {rouge['rougeL']:>10.3f}")

Summary                  BLEU    ROUGE-1    ROUGE-L
----------------------------------------------------
Good summary            0.225      0.706      0.588
Bad summary             0.000      0.256      0.256
Synonym summary         0.133      0.400      0.400


**What to notice:**
- The **good summary** scores well — it uses similar words to the reference.
- The **synonym summary** may score poorly on BLEU despite being semantically correct.
- The **bad summary** scores near zero, as expected.
- This illustrates the core limitation: BLEU and ROUGE reward word overlap, not meaning.
- **Hallucination blind spot**: a fluent, confident summary with entirely wrong facts would score similarly to the synonym summary if it shares domain vocabulary with the reference. BLEU and ROUGE cannot detect fabricated content.

---

### 1b — BERTScore

BERTScore (Zhang et al., 2020) addresses the synonym problem. Instead of counting matching words, it uses contextual embeddings from BERT to compute semantic similarity. A paraphrase scores highly even if it shares no words with the reference.

In [4]:
# Run BERTScore on the same three candidates from 1a
references_list = [reference] * len(candidates)
candidates_list = list(candidates.values())
candidate_names = list(candidates.keys())

P, R, F1 = bert_score_fn(candidates_list, references_list, lang="en", verbose=False)

print(f"{'Summary':<20} {'BERTScore F1':>14}")
print("-" * 36)
for name, f1 in zip(candidate_names, F1):
    print(f"{name:<20} {f1.item():>14.3f}")

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Summary                BERTScore F1
------------------------------------
Good summary                  0.965
Bad summary                   0.883
Synonym summary               0.951


**What to notice:**
- The **synonym summary** should now score similarly to (or better than) the good summary — BERTScore rewards semantic equivalence.
- The **bad summary** still scores low, even with BERTScore.
- This shows the value of embedding-based metrics for tasks where paraphrasing is acceptable.
- **Hallucination blind spot**: BERTScore measures *semantic similarity to the reference*, not factual accuracy. A hallucinated summary that uses plausible, in-domain language (e.g. "Stanford researchers" instead of "MIT researchers", "battery" still present) can still achieve a moderate-to-high BERTScore — because the embedding space groups related concepts together regardless of whether the facts are correct. Look for this in section 1d.

---

### 1c — Perplexity with GPT-2

**Perplexity** measures how surprised a language model is by a piece of text. Lower perplexity = the model finds the text more probable = the text is more fluent.

Perplexity does **not** measure factual accuracy — a fluent lie scores better than an awkward truth.

We use GPT-2 (a small, open model) as the language model for this demo.

In [5]:
print("Loading GPT-2...")
tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
gpt2_model = GPT2LMHeadModel.from_pretrained("gpt2")
gpt2_model.eval()
print("GPT-2 loaded.")

def compute_perplexity(text):
    encodings = tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        outputs = gpt2_model(**encodings, labels=encodings["input_ids"])
    return torch.exp(outputs.loss).item()

examples = {
    "Fluent":                      "The restaurant was beautifully decorated and the service was excellent.",
    "Disfluent":                   "Restaurant the was decorated beautifully excellent service the was and.",
    "Factually wrong but fluent":  "The Eiffel Tower is located in London, where it was built in 1889.",
}

print(f"\n{'Text type':<35} {'Perplexity':>12}")
print("-" * 49)
for label, text in examples.items():
    print(f"{label:<35} {compute_perplexity(text):>12.1f}")

Loading GPT-2...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT-2 loaded.

Text type                             Perplexity
-------------------------------------------------


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Fluent                                      22.2
Disfluent                                 1138.5
Factually wrong but fluent                   9.8


**What to notice:**
- The disfluent sentence has much higher perplexity — GPT-2 finds it improbable.
- The factually wrong but fluent sentence has **low perplexity** — it reads naturally.
- Perplexity captures fluency, not factual accuracy. A grammatically smooth false statement scores better than a clumsy true one.
- **Hallucination blind spot**: this is perplexity's most important limitation. A confident, well-phrased hallucination — "Stanford researchers unveiled a battery that charges in 30 seconds" — will score *lower* perplexity (more fluent) than a garbled true statement. Perplexity is useful for detecting incoherent outputs, but it cannot distinguish a fluent truth from a fluent lie.

---

### 1d — End-to-End: Generate a Dataset and Score with Classical Metrics

*(Requires GitHub Token — uses the `chat()` function defined in the setup cell above)*

The previous sections demonstrated each metric on hand-crafted examples. Now let's build a complete evaluation pipeline:

1. Use the LLM to generate a summarisation dataset (source articles + reference summaries)
2. Run the LLM to produce a candidate summary for each source
3. Score every candidate with BLEU, ROUGE, and BERTScore
4. Report results in a side-by-side comparison table

This is the pattern you would use to benchmark a summarisation prompt or compare two model variants.

In [6]:
import json
import pathlib

# --- Step 1: Generate a summarisation dataset ---
generate_prompt = """Create a JSON array of 8 summarisation test cases.
Each object must have exactly three keys:
- "topic": a short label (e.g. "technology", "health", "environment")
- "source": a paragraph of 4-5 sentences on that topic
- "reference": an ideal one-sentence summary of the source

Vary topics across: technology, health, environment, business, science, sport.
Include 2 examples where the key fact is a specific number or statistic.

Return ONLY a valid JSON array. No explanation, no markdown fences."""

print("Generating summarisation dataset...")
raw = chat(generate_prompt, temperature=0.7, max_tokens=2000)

# Strip markdown fences if the model adds them
cleaned = raw.strip()
if "```" in cleaned:
    parts = cleaned.split("```")
    cleaned = parts[1]
    if cleaned.startswith("json"):
        cleaned = cleaned[4:]
cleaned = cleaned.strip()

summ_dataset = json.loads(cleaned)
print(f"Generated {len(summ_dataset)} LLM examples")

# --- Append 2 handcrafted hallucination examples ---
# These have the candidate pre-set to a hallucinated summary so they are NOT
# overwritten in Step 2. They are flagged with "hallucinated": True so we can
# highlight them in the scoring tables.

HALLUCINATED_EXAMPLES = [
    {
        "topic": "science [HALLUCINATED]",
        "source": (
            "Researchers at MIT have developed a new battery technology that charges in under "
            "2 minutes and holds three times more energy than conventional lithium-ion batteries. "
            "The breakthrough uses a novel solid-state electrolyte and could enable electric "
            "vehicles to travel over 1,000 km on a single charge. "
            "Commercial production is expected within 5 years."
        ),
        "reference": (
            "MIT researchers created a fast-charging battery with 3x the energy density of "
            "lithium-ion, potentially enabling EVs to travel over 1,000 km per charge, "
            "with commercial production expected in 5 years."
        ),
        # Hallucination: wrong institution, wrong charge time, fabricated price, wrong timeline
        "candidate": (
            "Stanford researchers unveiled a solar-powered battery that charges in 30 seconds "
            "and lasts a decade without replacement, expected commercially next year for under $100."
        ),
        "hallucinated": True,
    },
    {
        "topic": "health [HALLUCINATED]",
        "source": (
            "A new study published in The Lancet found that daily consumption of 30 grams of nuts "
            "reduces the risk of heart disease by 21%. The research tracked 120,000 participants "
            "over 15 years across 12 countries. Scientists recommend replacing processed snacks "
            "with a small handful of mixed nuts as part of a heart-healthy diet."
        ),
        "reference": (
            "A 15-year Lancet study of 120,000 people found that eating 30 grams of nuts daily "
            "reduces heart disease risk by 21%, with scientists recommending nuts as a snack replacement."
        ),
        # Hallucination: wrong institution, fabricated outcome, wrong study duration
        "candidate": (
            "Harvard researchers found that eating two cups of almonds daily eliminates heart "
            "disease entirely, with a 3-month trial showing complete reversal of arterial blockages."
        ),
        "hallucinated": True,
    },
]

summ_dataset.extend(HALLUCINATED_EXAMPLES)
print(f"Added {len(HALLUCINATED_EXAMPLES)} handcrafted hallucination examples")
print(f"Total dataset size: {len(summ_dataset)}\n")

SUMM_DATASET_PATH = pathlib.Path("summ_dataset.json")
with open(SUMM_DATASET_PATH, "w") as f:
    json.dump(summ_dataset, f, indent=2)
print(f"Saved to {SUMM_DATASET_PATH}\n")

for item in summ_dataset[:3]:
    print(f"[{item['topic']}]")
    print(f"  Source:    {item['source'][:80]}...")
    print(f"  Reference: {item['reference'][:80]}")
    print()
print("... (hallucinated examples appended at end)")

Generating summarisation dataset...
Generated 8 LLM examples
Added 2 handcrafted hallucination examples
Total dataset size: 10

Saved to summ_dataset.json

[technology]
  Source:    Artificial intelligence has rapidly advanced over the past decade, transforming ...
  Reference: AI has advanced significantly, impacting industries and raising ethical and empl

[health]
  Source:    A recent study shows that regular physical activity can reduce the risk of cardi...
  Reference: Regular exercise can decrease cardiovascular risk by up to 30%, highlighting the

[environment]
  Source:    Global temperatures have increased by approximately 1.2 degrees Celsius since th...
  Reference: Since the 19th century, global temperatures have risen 1.2°C, causing more extre

... (hallucinated examples appended at end)


In [7]:
# --- Step 2: Generate candidate summaries ---
# Items with a pre-set candidate (the hallucinated examples) are skipped —
# their candidate is already the planted hallucination and must not be overwritten.

SUMM_PROMPT = (
    "Summarise the following text in exactly one sentence.\n\n"
    "Text: {source}\nSummary:"
)

print("Generating candidate summaries...")
for item in summ_dataset:
    if "candidate" not in item:
        item["candidate"] = chat(SUMM_PROMPT.format(source=item["source"]), temperature=0.3, max_tokens=80)

n_generated = sum(1 for item in summ_dataset if not item.get("hallucinated"))
print(f"Generated {n_generated} LLM summaries  |  {sum(1 for i in summ_dataset if i.get('hallucinated'))} hallucinated (pre-set)\n")

for item in summ_dataset[:3]:
    print(f"[{item['topic']}]")
    print(f"  Reference: {item['reference'][:80]}")
    print(f"  Candidate: {item['candidate'][:80]}")
    print()
print("...")
for item in summ_dataset[-2:]:
    tag = " *** HALLUCINATED ***" if item.get("hallucinated") else ""
    print(f"[{item['topic']}]{tag}")
    print(f"  Reference: {item['reference'][:80]}")
    print(f"  Candidate: {item['candidate'][:80]}")
    print()

Generating candidate summaries...
Generated 8 LLM summaries  |  2 hallucinated (pre-set)

[technology]
  Reference: AI has advanced significantly, impacting industries and raising ethical and empl
  Candidate: Artificial intelligence's rapid development has transformed industries and drive

[health]
  Reference: Regular exercise can decrease cardiovascular risk by up to 30%, highlighting the
  Candidate: Regular physical activity significantly lowers cardiovascular risk, and experts 

[environment]
  Reference: Since the 19th century, global temperatures have risen 1.2°C, causing more extre
  Candidate: Global temperature rise of about 1.2°C since the 19th century has intensified we

...
[science [HALLUCINATED]] *** HALLUCINATED ***
  Reference: MIT researchers created a fast-charging battery with 3x the energy density of li
  Candidate: Stanford researchers unveiled a solar-powered battery that charges in 30 seconds

[health [HALLUCINATED]] *** HALLUCINATED ***
  Reference: A 15-year 

In [8]:
# --- Step 3: Score every candidate with BLEU, ROUGE, and BERTScore ---

candidates_list = [item["candidate"] for item in summ_dataset]
references_list = [item["reference"] for item in summ_dataset]

# BLEU and ROUGE (per item)
bleu_scores, rouge1_scores, rougeL_scores = [], [], []
for cand, ref in zip(candidates_list, references_list):
    b = bleu_metric.compute(predictions=[cand], references=[[ref]])
    r = rouge_metric.compute(predictions=[cand], references=[ref])
    bleu_scores.append(b["bleu"])
    rouge1_scores.append(r["rouge1"])
    rougeL_scores.append(r["rougeL"])

# BERTScore (batched)
_, _, bert_F1 = bert_score_fn(candidates_list, references_list, lang="en", verbose=False)
bert_scores = bert_F1.tolist()

# Display results — flag hallucinated rows with ⚠
print(f"{'Topic':<26} {'BLEU':>7} {'R-1':>7} {'R-L':>7} {'BERT-F1':>9}")
print("-" * 65)
for item, bleu, r1, rl, bert in zip(summ_dataset, bleu_scores, rouge1_scores, rougeL_scores, bert_scores):
    flag = " \u26a0" if item.get("hallucinated") else ""
    print(f"{item['topic']:<26}{flag} {bleu:>7.3f} {r1:>7.3f} {rl:>7.3f} {bert:>9.3f}")

avg = lambda lst: sum(lst) / len(lst)
print("-" * 65)
print(f"{'Average (all)':<26} {avg(bleu_scores):>7.3f} {avg(rouge1_scores):>7.3f} {avg(rougeL_scores):>7.3f} {avg(bert_scores):>9.3f}")

# Separate averages: normal vs hallucinated
normal  = [(b, r1, rl, bert) for item, b, r1, rl, bert in zip(summ_dataset, bleu_scores, rouge1_scores, rougeL_scores, bert_scores) if not item.get("hallucinated")]
halluc  = [(b, r1, rl, bert) for item, b, r1, rl, bert in zip(summ_dataset, bleu_scores, rouge1_scores, rougeL_scores, bert_scores) if item.get("hallucinated")]
if halluc:
    print(f"\n{'Normal avg':<26} {avg([x[0] for x in normal]):>7.3f} {avg([x[1] for x in normal]):>7.3f} {avg([x[2] for x in normal]):>7.3f} {avg([x[3] for x in normal]):>9.3f}")
    print(f"{'Hallucinated avg \u26a0':<26} {avg([x[0] for x in halluc]):>7.3f} {avg([x[1] for x in halluc]):>7.3f} {avg([x[2] for x in halluc]):>7.3f} {avg([x[3] for x in halluc]):>9.3f}")

# Attach scores and save — reused in Part 3
for item, bleu, r1, rl, bert in zip(summ_dataset, bleu_scores, rouge1_scores, rougeL_scores, bert_scores):
    item["scores_classical"] = {"bleu": bleu, "rouge1": r1, "rougeL": rl, "bertscore_f1": bert}

with open(SUMM_DATASET_PATH, "w") as f:
    json.dump(summ_dataset, f, indent=2)
print(f"\nDataset with scores saved to {SUMM_DATASET_PATH}")

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Topic                         BLEU     R-1     R-L   BERT-F1
-----------------------------------------------------------------
technology                   0.203   0.432   0.432     0.936
health                       0.000   0.400   0.267     0.904
environment                  0.264   0.681   0.468     0.958
business                     0.167   0.490   0.449     0.940
science                      0.138   0.596   0.468     0.956
sport                        0.133   0.533   0.311     0.922
environment                  0.000   0.474   0.368     0.931
technology                   0.128   0.571   0.408     0.941
science [HALLUCINATED]     ⚠   0.000   0.172   0.138     0.891
health [HALLUCINATED]      ⚠   0.000   0.357   0.321     0.901
-----------------------------------------------------------------
Average (all)                0.103   0.471   0.363     0.928

Normal avg                   0.129   0.522   0.396     0.936
Hallucinated avg ⚠           0.000   0.265   0.230     0.896

Dataset 

**What to notice:**
- BLEU tends to score lower than ROUGE — it applies a brevity penalty and requires exact n-gram matches.
- BERTScore is more forgiving of paraphrasing — look for cases where BERTScore is noticeably higher than BLEU/ROUGE.
- Items with specific statistics may score lower if the LLM rephrases the number differently (e.g. "30 percent" vs "30%").
- **Hallucinated rows (⚠)**: compare the hallucinated summaries to normal ones. Classical metrics may not clearly distinguish them — the hallucinations use plausible, in-domain language, so BLEU/ROUGE/BERTScore can remain moderate even when the facts are entirely wrong. This is the core blind spot of classical metrics. The comparison against LLM-as-judge scores in Part 3 will make this concrete.
- The dataset is saved to `summ_dataset.json` with classical scores attached — Part 3 re-evaluates the same summaries using LLM-as-judge so you can compare both approaches directly.

---

## Part 2 — Benchmarks & Leaderboards

No code needed — this section is guided reading.

Classical metrics tell you how a model performs on your specific dataset. Leaderboards tell you how models compare on standardised benchmarks — useful for choosing a starting point, but a poor substitute for your own evaluation.

**Key leaderboards to explore:**

| Leaderboard | What it measures | URL |
|---|---|---|
| HF Open LLM Leaderboard | Open-source model benchmarks (MMLU, HellaSwag, etc.) | [huggingface.co/spaces/open-llm-leaderboard](https://huggingface.co/spaces/open-llm-leaderboard/open_llm_leaderboard) |
| LMSYS Chatbot Arena | Human preference (Elo rating from real user votes) | [arena.ai](https://arena.ai) |
| Artificial Analysis | Intelligence + speed + cost + latency | [artificialanalysis.ai](https://artificialanalysis.ai) |

**Questions to ask when reading a leaderboard:**
1. Is the benchmark saturated? (If top models all score 90%+, small differences are noise.)
2. What is *not* reported? (Labs choose which benchmarks to publish.)
3. Does strong benchmark performance predict strong performance on *my* task? (Usually not directly.)

**Key insight:** Leaderboards give you an evidence-based shortlist. They replace uninformed guessing — but they are proxies for your actual use case. The further your task is from standard benchmarks, the more critical your own evaluation becomes.

---

## Part 3 — Custom Eval Pipelines

*(Requires GitHub Token — set `GITHUB_TOKEN` in your environment)*

Classical metrics work well when you have reference text to compare against. For open-ended tasks — or when you want richer feedback than a single number — you need something more. This part builds a custom evaluation pipeline from first principles:

- **3a**: LLM-as-Judge — use a model to score another model's output
- **3b**: DIY eval loop — assemble the full pipeline on the dataset from Part 1d
- **3c**: Promptfoo — a CLI tool for prompt testing at scale
- **3d**: OpenAI Evals API — the managed version of the same pattern *(extension, requires OpenAI API key)*

### 3a — LLM-as-Judge

For open-ended tasks like summarisation, there is no single correct answer — classical metrics fall short when the candidate is correct but paraphrased differently from the reference. **LLM-as-judge** uses a strong model to evaluate another model's output against a rubric.

Pattern:
1. Define a rubric (scoring criteria, typically 1–5)
2. Send the source, candidate output, and rubric to the judge model
3. Ask for a score and justification in JSON format
4. Use `temperature=0` for the judge — you want consistent, deterministic scoring

We apply it to the summarisation dataset from Part 1d and compare the judge scores to the classical metrics we already computed.

In [9]:
import json as _json

SUMMARISATION_RUBRIC = """
5 - Captures all key facts, concise, no hallucinations or omissions
4 - Captures most key facts, minor omissions only
3 - Captures some key facts, notable gaps or imprecisions
2 - Mostly misses key facts or contains significant errors
1 - Completely wrong, irrelevant, or incoherent
"""

def llm_judge(source, candidate, rubric=SUMMARISATION_RUBRIC):
    """Use an LLM to score a summary against a rubric. Returns (score, justification)."""
    prompt = (
        f"You are an expert evaluator. Score the following summary using the rubric provided.\n\n"
        f"Source article: {source}\n"
        f"Summary: {candidate}\n"
        f"Rubric: {rubric}\n\n"
        'Respond with ONLY a JSON object: {"score": <int 1-5>, "justification": "<1 sentence>"}'
    )
    raw = chat(prompt, temperature=0, max_tokens=150)
    try:
        cleaned = raw.strip().strip("```json").strip("```").strip()
        data = _json.loads(cleaned)
        return data["score"], data["justification"]
    except _json.JSONDecodeError:
        return None, f"Parse error: {raw[:60]}"

print("llm_judge() defined.")

llm_judge() defined.


In [10]:
# Apply LLM-as-judge to the dataset and compare to classical metrics from Part 1d
# Hallucinated rows are flagged with ⚠ so you can see the contrast directly.

print("Scoring summaries with LLM-as-judge...")
print(f"{'Topic':<28} {'Judge/5':>8} {'BERT-F1':>9} {'ROUGE-1':>9}")
print("-" * 62)

for item in summ_dataset:
    score, justification = llm_judge(item["source"], item["candidate"])
    item["llm_judge_score"] = score
    bert  = item["scores_classical"]["bertscore_f1"]
    rouge = item["scores_classical"]["rouge1"]
    flag  = " \u26a0" if item.get("hallucinated") else ""
    print(f"{item['topic']:<28}{flag} {str(score):>8} {bert:>9.3f} {rouge:>9.3f}")

with open(SUMM_DATASET_PATH, "w") as f:
    _json.dump(summ_dataset, f, indent=2)
print("\nDataset updated with LLM-judge scores.")

Scoring summaries with LLM-as-judge...
Topic                         Judge/5   BERT-F1   ROUGE-1
--------------------------------------------------------------
technology                          4     0.936     0.432
health                              4     0.904     0.400
environment                         4     0.958     0.681
business                            4     0.940     0.490
science                             4     0.956     0.596
sport                               4     0.922     0.533
environment                         4     0.931     0.474
technology                          4     0.941     0.571
science [HALLUCINATED]       ⚠        2     0.891     0.172
health [HALLUCINATED]        ⚠        2     0.901     0.357

Dataset updated with LLM-judge scores.


**What to notice:**
- The two **hallucinated rows (⚠)** should score 1 or 2 from the judge — wrong institution, fabricated numbers, invented outcomes.
- Now look back at their classical metric scores from section 1d: BLEU/ROUGE/BERTScore likely rated them as *moderate*, not terrible. The hallucinations used plausible domain vocabulary, so classical metrics were fooled.
- This is the key contrast: **classical metrics saw acceptable output; the judge caught the fabrication**.
- For the normal summaries, judge and BERTScore should broadly agree — both reward semantically correct content. Disagreements in the normal rows (high BERTScore, lower judge) typically mean the summary is fluent but omits a key fact.

---

### 3b — DIY Eval Loop

The core pattern for any evaluation pipeline:
1. Load your dataset
2. Run the model on each input
3. Score each output
4. Aggregate and report

We run this on the summarisation dataset from Part 1d, using LLM-as-judge as the scorer — then show a combined table with both classical and judge scores side by side.

In [ ]:
import json
import pathlib

# Load the dataset — already contains source, reference, candidate, and classical scores
summ_dataset = json.loads(pathlib.Path("summ_dataset.json").read_text())

EVAL_SUMM_PROMPT = (
    "Summarise the following text in exactly one sentence.\n\n"
    "Text: {source}\nSummary:"
)

print("Running DIY eval loop...")
eval_results = []

for item in summ_dataset:
    # Generate a fresh candidate at temperature=0 for reproducibility
    print("..", end="", flush=True) # progress bar
    candidate = chat(EVAL_SUMM_PROMPT.format(source=item["source"]), temperature=0.0, max_tokens=80)
    score, justification = llm_judge(item["source"], candidate)
    eval_results.append({
        "topic":         item["topic"],
        "candidate":     candidate,
        "judge_score":   score,
        "bertscore_f1":  item["scores_classical"]["bertscore_f1"],
        "rouge1":        item["scores_classical"]["rouge1"],
        "bleu":          item["scores_classical"]["bleu"],
        "justification": justification,
    })

# Combined results table
print(f"\n{'Topic':<15} {'Judge/5':>8} {'ROUGE-1':>9} {'BERT-F1':>9}")
print("-" * 50)
for r in eval_results:
    print(f"{r['topic']:<15} {str(r['judge_score']):>8} {r['rouge1']:>9.3f} {r['bertscore_f1']:>9.3f}")

valid = [r["judge_score"] for r in eval_results if r["judge_score"] is not None]
print(f"\nAverage LLM-judge score: {sum(valid)/len(valid):.1f} / 5")

print("\nSample justifications:")
for r in eval_results[:3]:
    print(f"  [{r['topic']}] Score {r['judge_score']}: {(r['justification'] or '')[:80]}")

**What to notice:**
- This is the fundamental eval loop pattern. Everything else — Promptfoo, the OpenAI Evals API — is a variation of: run → score → aggregate.
- The classical scores (ROUGE, BERTScore) come from Part 1d and reflect the *same summaries* scored a different way. Compare the rankings: do all three metrics agree on which summaries are best?
- `temperature=0.0` in the model call gives consistent, reproducible outputs — important for evaluation.
- The justifications explain *why* a summary scored the way it did, which a number alone cannot.

---

### 3c — Promptfoo

[Promptfoo](https://github.com/promptfoo/promptfoo) is an open-source CLI tool for prompt testing and model comparison. You define test cases in YAML, run them against one or more models, and get a visual comparison report.

It is particularly useful for:
- Comparing the same prompt across multiple models side by side
- Regression testing when you update a prompt
- Projects where you need a shareable, visual evaluation report

**A working example is in this session's `getting-started/` folder** — it tests a translation prompt against a local Ollama model with no API key required.

```bash
# From material/Session 7/getting-started/
npx promptfoo eval
npx promptfoo view
```

The `promptfooconfig.yaml` shows the core pattern: prompts → providers → test cases with assertions.

In [ ]:
# Display the promptfoo config from the getting-started example
with open("../getting-started/promptfooconfig.yaml") as f:
    print(f.read())

**When to use Promptfoo vs. a Python eval loop:**
- Use **Promptfoo** when you want to compare multiple models or prompt variants visually, or need a shareable report with minimal code.
- Use a **Python eval loop** (section 3b) when you need custom scoring logic, integration with your own data pipeline, or fine-grained control.

Both are complementary — Promptfoo for quick exploration, Python loops for production eval pipelines.

---

### 3d — OpenAI Evals API *(Optional Extension)*

The OpenAI Evals API is the managed version of the DIY pipeline from section 3b. You define the same pattern — a prompt template and grading criteria — and OpenAI handles execution, storage, and aggregation at scale.

We use the summarisation dataset from Part 1d and score it with a `label_model` grader — the managed equivalent of `llm_judge()` from section 3a.

**Requires:** `OPENAI_API_KEY` in your `.env` file.

In [ ]:
# OPTIONAL — requires OPENAI_API_KEY in .env
from dotenv import load_dotenv
import os

load_dotenv(".env", override=True)

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
OPENAI_ORG_ID = os.environ.get("OPENAI_ORG_ID", "")
OPENAI_PROJECT_ID = os.environ.get("OPENAI_PROJECT_ID", "")
if not OPENAI_API_KEY:
    raise ValueError(
        "OPENAI_API_KEY not found. "
        "Add OPENAI_API_KEY=sk-... to your .env file and restart the kernel."
    )

from openai import OpenAI as OpenAIClient
oai = OpenAIClient(
    api_key=OPENAI_API_KEY,
    organization=OPENAI_ORG_ID or None,
    project=OPENAI_PROJECT_ID or None,
)
print("OpenAI client ready.")

#### How the OpenAI Evals API works

The API separates **definition** from **execution** — two distinct calls:

1. **`oai.evals.create()`** — defines the item schema and grading criteria. No model or data here.
2. **`oai.evals.runs.create(eval_id, ...)`** — ties the eval to a model, a prompt template, and a dataset file.

This means you can re-run the same eval definition against different models or datasets without redefining the graders each time.

**Step 1 — Upload the dataset.** The Evals API reads data from a JSONL file stored in OpenAI's Files service. Each line is one test case: `{"source": "...", "reference": "..."}`.

In [ ]:
import json
import pathlib
import time

# Load dataset from Part 1d
summ_dataset = json.loads(pathlib.Path("summ_dataset.json").read_text())
print(f"Loaded {len(summ_dataset)} items")

# Write as JSONL — each line must be {"item": {...}} when using type:"custom" evals.
# The "item" wrapper is required by the OpenAI Evals schema validation.
jsonl_path = pathlib.Path("summ_dataset.jsonl")
with open(jsonl_path, "w") as f:
    for item in summ_dataset:
        row = {"item": {"source": item["source"], "reference": item["reference"]}}
        f.write(json.dumps(row) + "\n")

with open(jsonl_path, "rb") as f:
    uploaded_file = oai.files.create(file=f, purpose="evals")

print(f"File uploaded — ID: {uploaded_file.id}")
print(f"Status: {uploaded_file.status}")

In [ ]:
# Load uploaded file metadata
uploaded_file = oai.files.retrieve("file-F6NDUWBToJR7G6j7NY6t7b")
print(f"File status: {uploaded_file.status}")
print(f"File id: {uploaded_file.id}")
print(f"Bytes: {uploaded_file.bytes}")

# Print content from OpenAI Files storage
raw_bytes = oai.files.content(uploaded_file.id).read()
text = raw_bytes.decode("utf-8")
print("\n--- File Content ---\n")
print(text)


In [ ]:
# --- Step 2: Create the eval definition ---
# The eval definition captures the item schema (what fields each row has)
# and the grading criteria. The model and prompts come in Step 3 when you create a run.

eval_def = oai.evals.create(
    name="anlp-session7-summarisation-eval",
    data_source_config={
        "type": "custom",
        "item_schema": {
            "type": "object",
            "properties": {
                "source":    {"type": "string"},
                "reference": {"type": "string"},
            },
            "required": ["source", "reference"],
        },
        "include_sample_schema": True,  # expect {{sample.output_text}} in grader
    },
    testing_criteria=[
        {
            "type": "label_model",
            "name": "summary_quality",
            "model": "gpt-4o-mini",
            "input": [
                {
                    "role": "user",
                    "content": (
                        "Source: {{item.source}}\n"
                        "Reference summary: {{item.reference}}\n"
                        "Candidate summary: {{sample.output_text}}\n\n"
                        "Score the candidate summary from 1 to 5:\n"
                        "5 = captures all key facts, concise, no hallucinations\n"
                        "4 = captures most key facts, minor omissions\n"
                        "3 = captures some key facts, notable gaps\n"
                        "2 = mostly misses key facts or contains errors\n"
                        "1 = completely wrong or incoherent\n\n"
                        "Reply with only the number."
                    ),
                }
            ],
            "passing_labels": ["4", "5"],
            "labels": ["1", "2", "3", "4", "5"],
        }
    ],
)

print(f"Eval definition created — ID: {eval_def.id}")

In [ ]:
# --- Step 3: Create a run ---
# A run ties the eval definition to a specific model, prompt template, and dataset.
# The file_id goes here — not in the eval definition.

run = oai.evals.runs.create(
    eval_def.id,
    name="session7-summarisation-run-1",
    data_source={
        "type": "completions",
        "source": {"type": "file_id", "id": uploaded_file.id},
        "input_messages": {
            "type": "template",
            "template": [
                {
                    "role": "user",
                    "content": (
                        "Summarise the following text in exactly one sentence.\n\n"
                        "Text: {{item.source}}\nSummary:"
                    ),
                }
            ],
        },
        "model": "gpt-4o-mini",
        "sampling_params": {"temperature": 0.0},
    },
)

print(f"Run created — ID: {run.id}")
print(f"Status:          {run.status}")

In [ ]:
# --- Step 4: Poll until the run completes ---
# Runs are asynchronous — poll until status is "completed" or "failed".
import time

print(f"Run ID: {run.id}")
print("Polling...")

MAX_WAIT = 180
poll_interval = 5
elapsed = 0
status = "unknown"

while elapsed < MAX_WAIT:
    current = oai.evals.runs.retrieve(run.id, eval_id=eval_def.id)
    status = getattr(current, "status", "unknown")
    print(f"  [{elapsed:3}s] status: {status}")
    if status == "completed":
        break
    if status in ("failed", "cancelled"):
        print(f"Run ended with status: {status}")
        break
    time.sleep(poll_interval)
    elapsed += poll_interval
else:
    print("Timeout — check the OpenAI dashboard.")

print(f"\nFinal status: {status}")

In [ ]:
# --- Step 5: Retrieve and display results ---
output_items = oai.evals.runs.output_items.list(run.id, eval_id=eval_def.id)

passed = 0
total  = 0

print(f"{'Source (truncated)':<40} {'Score':>7} {'Pass':>6}")
print("-" * 58)

for item in output_items.data:
    # datasource_item stores {"item": {"source": ..., "reference": ...}}
    inner = item.datasource_item.get("item", item.datasource_item)
    source_text = str(inner.get("source", ""))[:38]

    # result — Pydantic model with .passed (bool) and .score (float)
    result     = item.results[0] if item.results else None
    score      = f"{result.score:.0f}" if result else "?"
    passed_flag = result.passed if result else False

    icon = "\u2713" if passed_flag else "\u2717"
    passed += int(passed_flag)
    total  += 1
    print(f"{source_text:<40} {score:>7} {icon:>6}")

if total:
    print(f"\nManaged eval pass rate: {passed/total:.1%}  ({passed}/{total} scored 4 or 5)")

# Compare to DIY pipeline if section 3b was run in this session
try:
    valid = [r["judge_score"] for r in eval_results if r.get("judge_score") is not None]
    if valid:
        print(f"DIY pipeline avg score: {sum(valid)/len(valid):.1f} / 5  (from section 3b)")
except NameError:
    print("(Run section 3b first to see the DIY pipeline comparison)")

In [ ]:
# list all evals in the account and results
evals = oai.evals.list()
print(f"\nAll evals in account:")
for e in evals.data:
    print(f"  Eval ID: {e.id}  |  Name: {e.name}  |  Created: {e.created_at}")

# list eval runs for first eval in list and their results
if evals.data:
    eval_id = evals.data[0].id
    runs = oai.evals.runs.list(eval_id)
    print(f"\nRuns for eval {eval_id}:")
    for r in runs.data:
        print(f"  Name: {r.name}  |  Status: {r.status}")

**What to notice:**
- The `label_model` grader is the managed version of `llm_judge()` — same rubric, same model, same pattern.
- `passing_labels: ["4", "5"]` defines the threshold — equivalent to "score >= 4" in the DIY pipeline.
- **Why the managed eval scores higher than the DIY pipeline**: the managed eval generates *fresh* summaries from the source text using the model. It does **not** use the pre-seeded hallucinated candidates from Part 1d — those only exist in our dataset file. So the hallucination blind spot you saw in classical metrics is not replicated here; the managed eval just generates new (likely correct) summaries.
- **The Evals API stores results**, enabling regression testing: re-run the same eval definition against a new model or updated prompt and compare pass rates directly.

| What | DIY (section 3b) | OpenAI Evals API (this section) |
|---|---|---|
| Prompt template | Python f-string | Mustache `{{item.field}}` |
| LLM judge | `llm_judge()` function | `label_model` grader |
| Uses planted hallucinations | Yes — from dataset file | No — generates fresh outputs |
| Classical metrics | `evaluate` library | Not built-in |
| Scale | Manual loop | Managed, stored, version-controlled |

---

## Summary — Three Tiers of LLM Evaluation

| Tier | What | Tools | When to use |
|---|---|---|---|
| **A — Classical metrics** | Surface and semantic overlap | BLEU, ROUGE, BERTScore, perplexity | Translation, summarisation benchmarks |
| **B — Leaderboards** | Standardised benchmark comparison | HF Leaderboard, LMSYS Arena, Artificial Analysis | Choosing a model, not evaluating your task |
| **C — Custom pipelines** | Task-specific scoring | DIY eval loop, LLM-as-judge, Promptfoo, OpenAI Evals | Your real project — AT2 and beyond |

**Which approach for your AT2 project?**
- If your task has reference text (summarisation): start with ROUGE + BERTScore, then add LLM-as-judge for richer feedback
- If your task is open-ended (QA, generation): LLM-as-judge with a well-defined rubric is the right default
- Combine approaches — classical metrics for speed and reproducibility, LLM-as-judge for quality and nuance

**Remember:** Benchmark scores tell you about general capability. Your own eval tells you about your task.